# Sonoran AI Consulting, LLC | Grant Prospect Database
### Database overview, data quality, and funding prospect analysis

This notebook provides a reproducible overview of Sonoran AI Consulting's grant prospect database. It is designed to help reviewers understand the dataset, inspect prospect records, assess data completeness, and explore funding opportunities by type, status, fit score, and deadline information.

**Scope note:** This notebook describes the fields present in the database. It does not independently verify whether a funding opportunity is currently open or whether Sonoran AI Consulting is eligible. Confirm time-sensitive details with the original funder before acting on them.

**How to use:** Place the grant prospect CSV in the repository root or in `data/`, then run the notebook from top to bottom.


## 1. Setup and data loading

The source notebook refers to `Edited copy_Sonoran AI Grant Prospect Database1.csv`. The loader below checks that filename and common alternatives in the repository root and `data/` directory. If your CSV uses a different name or location, update `DATA_CANDIDATES`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Keep charts consistent and readable.
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["figure.autolayout"] = True

DATA_CANDIDATES = [
    Path("Edited copy_Sonoran AI Grant Prospect Database1.csv"),
    Path("Sonoran AI Grant Prospect Database1.csv"),
    Path("grant_prospect_database.csv"),
    Path("data/Edited copy_Sonoran AI Grant Prospect Database1.csv"),
    Path("data/Sonoran AI Grant Prospect Database1.csv"),
    Path("data/grant_prospect_database.csv"),
]

data_path = next((path for path in DATA_CANDIDATES if path.exists()), None)

if data_path is None:
    candidates = "\\n".join(f"- {path}" for path in DATA_CANDIDATES)
    raise FileNotFoundError(
        "Grant prospect CSV not found. Place the CSV in the repository root or data/ "
        "and update DATA_CANDIDATES if needed. Checked:\\n" + candidates
    )

grants_df = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Dataset shape: {grants_df.shape[0]:,} rows × {grants_df.shape[1]:,} columns")
display(grants_df.head())


## 2. Database snapshot

These headline metrics are calculated directly from the loaded CSV each time the notebook runs. Blank values are treated as missing for the completeness metrics.


In [ ]:
# Normalize column labels for reliable lookups while preserving the original data.
grants_df.columns = [str(col).strip() for col in grants_df.columns]

def find_column(expected_name):
    expected = expected_name.strip().casefold()
    return next((col for col in grants_df.columns if col.strip().casefold() == expected), None)

fund_col = find_column("Funder")
program_col = find_column("Program")
type_col = find_column("Type")
amount_col = find_column("Amount Range")
deadline_col = find_column("Deadline")
fit_col = find_column("Fit Score")
status_col = find_column("Status")

snapshot = {
    "Prospect records": len(grants_df),
    "Database fields": grants_df.shape[1],
    "Distinct funders": grants_df[fund_col].nunique(dropna=True) if fund_col else np.nan,
    "Distinct programs": grants_df[program_col].nunique(dropna=True) if program_col else np.nan,
    "Records with a funding amount/range": grants_df[amount_col].notna().sum() if amount_col else np.nan,
    "Records with a deadline/cycle entry": grants_df[deadline_col].notna().sum() if deadline_col else np.nan,
    "Records with a status": grants_df[status_col].notna().sum() if status_col else np.nan,
}
display(pd.DataFrame({"Metric": snapshot.keys(), "Value": snapshot.values()}))


## 3. Data dictionary

The current database schema includes the following fields. The definitions below are practical interpretations of the field names; confirm any organization-specific scoring or status conventions with the person maintaining the database.


In [ ]:
data_dictionary = [
    ("#", "Record or list number"),
    ("Funder", "Funding organization or grantmaker"),
    ("Program", "Named grant, funding program, or opportunity"),
    ("Type", "Funder/opportunity category, such as Federal, Foundation, Corporate, or State"),
    ("Amount Range", "Reported funding amount, range, pool, or award description"),
    ("Deadline", "Reported deadline, rolling cycle, or monitoring note"),
    ("Eligibility / Alignment Notes", "Notes on eligibility, strategic alignment, and follow-up considerations"),
    ("Fit Score", "Internal fit score; interpretation and scoring rubric should be documented separately"),
    ("Status", "Internal tracking status for the prospect"),
]
schema_df = pd.DataFrame(data_dictionary, columns=["Field", "Working definition"])
schema_df = schema_df[schema_df["Field"].isin(grants_df.columns)]
display(schema_df)


## 4. Data quality review

This section identifies missing values, duplicate rows, and repeated funder/program names. These are review flags, not automatic proof of an error: some opportunities may legitimately share a funder or have incomplete information while research is in progress.


In [ ]:
quality_summary = pd.DataFrame({
    "Missing values": grants_df.isna().sum(),
    "Missing %": (grants_df.isna().mean() * 100).round(1),
    "Distinct values": grants_df.nunique(dropna=True),
    "Data type": grants_df.dtypes.astype(str),
}).sort_values("Missing %", ascending=False)
display(quality_summary)

print(f"Exact duplicate rows: {grants_df.duplicated().sum()}")

duplicate_keys = [col for col in [fund_col, program_col] if col]
if len(duplicate_keys) == 2:
    repeated = grants_df[
        grants_df.duplicated(subset=duplicate_keys, keep=False)
    ].sort_values(duplicate_keys)
    print(f"Rows with repeated funder + program combination: {len(repeated)}")
    display(repeated if not repeated.empty else pd.DataFrame(columns=grants_df.columns))


## 5. Prospect landscape by type

The chart summarizes the categories recorded in the `Type` field. Category labels reflect the source database and should not be interpreted as a verification of the funder's legal or operational status.


In [ ]:
if type_col:
    type_counts = grants_df[type_col].fillna("Missing / not recorded").value_counts()
    display(type_counts.rename_axis("Type").reset_index(name="Prospect count"))

    ax = type_counts.sort_values().plot(kind="barh")
    ax.set_title("Prospects by type")
    ax.set_xlabel("Number of records")
    ax.set_ylabel("")
    plt.show()
else:
    print("No 'Type' column found; skipping this analysis.")


## 6. Status and pipeline review

Status is treated as an internal tracking field. The chart helps identify how the current prospect list is distributed across recorded statuses; it does not independently establish whether a grant is open.


In [ ]:
if status_col:
    status_counts = grants_df[status_col].fillna("Missing / not recorded").astype(str).str.strip()
    status_counts = status_counts.replace("", "Missing / not recorded").value_counts()
    display(status_counts.rename_axis("Status").reset_index(name="Prospect count"))

    ax = status_counts.sort_values().plot(kind="barh")
    ax.set_title("Prospects by tracking status")
    ax.set_xlabel("Number of records")
    ax.set_ylabel("")
    plt.show()
else:
    print("No 'Status' column found; skipping this analysis.")


## 7. Fit score distribution

The database contains an internal `Fit Score` field. This section shows its recorded distribution without assuming a scale or scoring rubric that is not documented in the data. Add a short description of the scale and criteria to the repository documentation before using it for formal prioritization.


In [ ]:
if fit_col:
    fit_numeric = pd.to_numeric(grants_df[fit_col], errors="coerce")
    display(fit_numeric.describe().rename("Fit Score summary").to_frame())

    score_counts = fit_numeric.value_counts(dropna=False).sort_index()
    score_counts.index = ["Missing / not numeric" if pd.isna(x) else x for x in score_counts.index]
    display(score_counts.rename_axis("Fit score").reset_index(name="Prospect count"))

    ax = fit_numeric.dropna().value_counts().sort_index().plot(kind="bar")
    ax.set_title("Recorded fit score distribution")
    ax.set_xlabel("Fit score")
    ax.set_ylabel("Number of prospects")
    plt.show()
else:
    print("No 'Fit Score' column found; skipping this analysis.")


## 8. Deadline and funding-cycle review

Deadline values may contain dates, rolling-cycle labels, or free-text monitoring notes. They should not be parsed as dates without checking the source wording. The first table summarizes the values as entered so that researchers can identify opportunities requiring follow-up.


In [ ]:
if deadline_col:
    deadline_summary = (
        grants_df[deadline_col]
        .fillna("Missing / not recorded")
        .astype(str).str.strip()
        .replace("", "Missing / not recorded")
        .value_counts()
        .rename_axis("Deadline / cycle entry")
        .reset_index(name="Prospect count")
    )
    display(deadline_summary)
else:
    print("No 'Deadline' column found; skipping this analysis.")

if amount_col:
    amount_summary = (
        grants_df[amount_col]
        .fillna("Missing / not recorded")
        .astype(str).str.strip()
        .replace("", "Missing / not recorded")
        .value_counts()
        .rename_axis("Reported amount / range")
        .reset_index(name="Prospect count")
    )
    display(amount_summary)


## 9. Searchable prospect review table

Use the cell below to filter prospects by a keyword found in any text field. This simple approach does not require additional widget libraries and works in standard Jupyter environments.


In [ ]:
search_term = ""  # Example: "Arizona", "AI", "foundation", or "rolling"

if search_term.strip():
    text_cols = grants_df.select_dtypes(include=["object", "string"]).columns
    mask = grants_df[text_cols].fillna("").astype(str).apply(
        lambda col: col.str.contains(search_term, case=False, regex=False)
    ).any(axis=1)
    filtered_df = grants_df.loc[mask]
else:
    filtered_df = grants_df

print(f"Showing {len(filtered_df):,} of {len(grants_df):,} prospect records.")
display(filtered_df)


## 10. Research takeaways and next steps

Use the results above to guide human review. Avoid treating missing information as a negative eligibility finding.

Recommended maintenance workflow:

1. Verify deadlines, eligibility, funding amounts, and application links against the funder's current source.
2. Record the source URL and date last checked for each prospect, if those fields are not already maintained elsewhere.
3. Document the `Fit Score` scale, scoring criteria, and who assigned the score.
4. Standardize status labels and define what each status means.
5. Review duplicate and incomplete records before prioritizing applications.
6. Re-run this notebook after each database update.

### Interpretation limits

- The analysis is descriptive and reflects only the loaded CSV.
- Free-text funding ranges are not converted into numeric values because entries may describe pools, ranges, or non-cash support.
- Deadline text is summarized as recorded and is not a live deadline verification.
- Fit scores and status labels are internal fields; their meaning depends on the team's documented process.
- This notebook does not determine legal eligibility or guarantee funding suitability.


## Appendix: reproducibility

Record the source file name and the date the database was last checked in the repository's documentation. Consider adding source URLs and `Last verified` fields to the database to support ongoing maintenance.


In [ ]:
print("Source file:", data_path)
print("Rows:", grants_df.shape[0])
print("Columns:", grants_df.shape[1])
print("Pandas version:", pd.__version__)
